In [ ]:
# ============================================================
# Cell 1 – Imports & Settings
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import xgboost as xgb
from lifelines.utils import concordance_index
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("NumPy     :", np.__version__)
print("XGBoost   :", xgb.__version__)
print("scikit-learn versions will be printed after import checks")

In [ ]:
# ============================================================
# Cell 2 – Load Data
# ============================================================
cols = ['unit_number', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
       [f'sensor_{i}' for i in range(1, 22)]

train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
test  = pd.read_csv('test_FD001.txt',  sep=r'\s+', header=None, names=cols)
rul_test = pd.read_csv('RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL'])

print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
print(f"RUL   shape : {rul_test.shape}")

In [ ]:
# ============================================================
# Cell 3 – Create RUL target (piecewise linear)
# ============================================================
max_cycle = train.groupby('unit_number')['time_cycles'].max().reset_index()
max_cycle.columns = ['unit_number', 'max_cycle']

train = train.merge(max_cycle, on='unit_number', how='left')
train['RUL'] = train['max_cycle'] - train['time_cycles']

# Standard piecewise-linear labeling used on FD001
RUL_CLIP = 125
train['RUL'] = train['RUL'].clip(upper=RUL_CLIP)

print("RUL statistics after clipping:")
print(train['RUL'].describe())

In [ ]:
# ============================================================
# Cell 4 – Feature selection
# ============================================================
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]

variances = train[sensor_cols].var()
constant_sensors = variances[variances < 1e-6].index.tolist()
print("Dropping constant sensors:", constant_sensors)

useful_sensors = [c for c in sensor_cols if c not in constant_sensors]
print("Useful sensors:", useful_sensors)

feature_cols = useful_sensors

In [ ]:
# ============================================================
# Cell 5 – Prepare train / test matrices
# ============================================================
X_train = train[feature_cols].copy()
y_train = train['RUL'].copy()

# Test set = last cycle of each engine
last_cycle_test = test.loc[test.groupby('unit_number')['time_cycles'].idxmax()]
X_test = last_cycle_test[feature_cols].reset_index(drop=True)
y_test = rul_test['RUL'].values

print(f"Training samples : {X_train.shape[0]}")
print(f"Test engines     : {X_test.shape[0]}")
print(f"Features         : {len(feature_cols)}")

In [ ]:
# ============================================================
# Cell 6 – Scale features
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# ============================================================
# Cell 7 – Train XGBoost
# ============================================================
model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.5,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.15, random_state=42
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Model trained successfully.")

In [ ]:
# ============================================================
# Cell 8 – Evaluate
# ============================================================
y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1))

score = nasa_score(y_test, y_pred)
c_index = concordance_index(y_test, y_pred)   # correct polarity for RUL

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"RMSE        : {rmse:.2f}")
print(f"MAE         : {mae:.2f}")
print(f"NASA Score  : {score:.1f}   (lower is better)")
print(f"Concordance : {c_index:.3f}  (higher is better)")
print("=" * 50)

In [ ]:
# ============================================================
# Cell 9 – Feature importance plot
# ============================================================
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis')
plt.title('XGBoost Feature Importance – FD001')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 10 – Actual vs Predicted
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7, edgecolors='k')
plt.plot([0, max(y_test.max(), y_pred.max())], 
         [0, max(y_test.max(), y_pred.max())], 'r--', lw=2)
plt.xlabel('True RUL')
plt.ylabel('Predicted RUL')
plt.title(f'True vs Predicted RUL  (RMSE = {rmse:.1f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 11 – Save artifacts (IMPORTANT)
# ============================================================
import sklearn

artifacts = {
    'model': model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'rul_clip': RUL_CLIP,
    'versions': {
        'numpy': np.__version__,
        'sklearn': sklearn.__version__,
        'xgboost': xgb.__version__,
        'joblib': joblib.__version__
    }
}

joblib.dump(artifacts, 'rul_model_fd001.joblib')
print("Model saved to 'rul_model_fd001.joblib'")
print("Versions used:")
print(artifacts['versions'])